# E7 Notebook 30C — Safe model checkpoint recovery

Notebook 30B verified the existing draft release but stopped when its freshly built harmless preflight ZIP differed from the earlier archived preflight ZIP. Notebook 30C verifies and executes the original frozen Notebook 30B code by its Git blob SHA, while substituting only the Git-hash-fixed, deterministic V3 publisher. A new probe filename avoids any conflict with the older probe. The original seed-42 model, G3 protocol, model settings and all prior results are retained.


In [ ]:
from pathlib import Path
import hashlib,json,importlib.util
from urllib.request import urlopen

REPO="itsCodeBakery/rare-attack-spatiotemporal-ids"
PIN="eacb821f8b428432f04e8418df5d75b4437bd73f"
ROOT="research/E7_CICIDS2017"
NB_PATH=ROOT+"/notebooks/30b_e7_g3_recovery.ipynb"
NB_BLOB="8947e6980d7f7ce09a3a1de9f52aceaeb98c7884"
V3_PATH=ROOT+"/scripts/e7_g3_archive_v3.py"
V3_BLOB="033fbc2f7957f54bdaf7b6ad9ad6da6b26d8f619"

def pinned_bytes(path,digest):
    url="https://raw.githubusercontent.com/"+REPO+"/"+PIN+"/"+path
    with urlopen(url,timeout=90) as response:data=response.read()
    actual=hashlib.sha1(b"blob "+str(len(data)).encode()+b"\\x00"+data).hexdigest()
    assert actual==digest,(path,"Blob SHA mismatch")
    return data

SOURCE=json.loads(pinned_bytes(NB_PATH,NB_BLOB))
CELLS=[c for c in SOURCE["cells"] if c["cell_type"]=="code"]
assert len(CELLS)==4,"Original Notebook 30B changed code layout"
first="".join(CELLS[0]["source"])
assert "G3_OPENWORLD_DAYHOLDOUT_PROTOCOL_001.json" in first
assert 'COMMIT="bb63819d32330b8102f9a4ce4f136da67d598f46"' in first
assert "E7_G3_DAYHOLDOUT_001" in first

def run_verified_cell(index):
    source="".join(CELLS[index]["source"])
    code=compile(source,"verified_notebook30b_cell_"+str(index+1),"exec")
    exec(code,globals(),globals())

run_verified_cell(0)
v3_source=pinned_bytes(V3_PATH,V3_BLOB)
V3_CACHE=Path("/kaggle/working/E7_G3_PINNED_SOURCE_30C_001/e7_g3_archive_v3.py")
V3_CACHE.parent.mkdir(parents=True,exist_ok=True)
if V3_CACHE.exists(): assert V3_CACHE.read_bytes()==v3_source
else:V3_CACHE.write_bytes(v3_source)
spec=importlib.util.spec_from_file_location("g3_publisher_v3",V3_CACHE)
archive=importlib.util.module_from_spec(spec)
spec.loader.exec_module(archive)
assert archive.TAG=="e7-g3-openworld-dayholdout-001-private-models"
print("30C SOURCE VERIFIED; V3 DETERMINISTIC PROBE ACTIVE; ORIGINAL MODEL PATH UNCHANGED.")

## Preflight — verify private draft model storage before loading traffic


In [ ]:
run_verified_cell(1)


## Original dataset integrity and feature extraction


In [ ]:
run_verified_cell(2)


## Resume and verify previous model; then run remaining frozen folds


In [ ]:
run_verified_cell(3)
